# BGC-Argo DCM Pipeline
**End-to-end**: BGC-Argo Sprof NetCDF → `profileswithparandsurfchl.csv`

**What is computed per profile**
- DCM type: NO / DAM / DBM (revised 1.5x CHLA threshold, round-half-up, BBP 1.3x DAM/DBM split)
- DCM depth, CHLA at DCM, BBP700 at DCM, CHLA ratio, BBP ratio
- Mixed Layer Depth (TEOS-10 sigma0, 0.03 kg/m³ from 10 dbar)
- Nitracline depth (Cornec 2021: surface mean + 1 µmol NO₃/m³)
- 26 °C isotherm depth
- Monsoon phase (NE / Spring IM / SW / Fall IM)
- Sub-basin from lat/lon bounding boxes (SCTR, AS, BoB, EIO, SIO)
- Matched surface chlorophyll and PAR from monthly satellite NetCDF

**Run order**: execute cells top to bottom. Update paths in the Configuration cell if needed.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
import gsw
from decimal import Decimal, ROUND_HALF_UP

In [ ]:
FLOAT_DIRS = [
    "/Users/ishita/Desktop/Bioargov3/fixed_Arabian_Sea_chl_bioargo",
    "/Users/ishita/Desktop/Bioargov3/fixed_Bay_of_Bengal_chl_bioargo",
    "/Users/ishita/Desktop/Bioargov3/fixed_Equatorial_IO_chl_bioargo",
    "/Users/ishita/Desktop/Bioargov3/fixed_Southern_Indian_Ocean_chl_bioargo",
]

CHL_NC  = "/Users/ishita/Desktop/Bioargov3/io_data/chl_monthly_2012_2024_IO.nc"
PAR_NC  = "/Users/ishita/Desktop/Bioargov3/io_data/par_monthly_2012_2024_IO.nc"
OUTPUT_CSV = "/Users/ishita/Desktop/Bioargov3/io_data/basin_csvs/profileswithparandsurfchl_fullrun.csv"

QC_PASS_SET             = {1, 2, 5}
TOP_250M_DEPTH          = 250.0
SMOOTHING_RES_THRESHOLD = 3.0
CHLA_SEARCH_MAX_DEPTH   = 300.0
DCM_SURFACE_DEPTH       = 15.0
BBP_SEARCH_WINDOW       = 20.0
MLD_DENSITY_THRESHOLD   = 0.03
NITRACLINE_SURFACE_DEPTH= 10.0
NITRACLINE_DELTA        = 1.0
ISOTHERM_THRESHOLD      = 26.0
DCM_CHLA_THRESHOLD      = 1.5
DBM_BBP_THRESHOLD       = 1.3

SUB_BASIN_BOXES = {
    "SCTR": (-10, -5,  50,  80),
    "AS":   (  5, 25,  50,  80),
    "BoB":  (  5, 25,  80, 100),
    "EIO":  ( -5,  5,  50, 100),
    "SIO":  (-30, -5,  50, 100),
}

In [ ]:
def qc_to_int(q):
    """
    Normalize an Argo QC flag to int.

    Parameters
    ----------
    q : bytes | str | int | float | None
        Raw QC value from NetCDF.

    Returns
    -------
    int
        QC as integer, or -1 when unparseable (-1 is never a valid Argo flag).
    """
    if q is None:
        return -1
    if isinstance(q, bytes):
        q = q.decode("utf-8", errors="ignore").strip()
        return int(q) if q.isdigit() else -1
    if isinstance(q, (np.floating, float)):
        return -1 if np.isnan(q) else int(q)
    if isinstance(q, (np.integer, int)):
        return int(q)
    q = str(q).strip()
    return int(q) if q.isdigit() else -1


def extract_var_with_fallback(ds, param, prof_idx):
    """
    Pull adjusted values, QC flags, and pressure for one parameter, accepting
    only D (delayed) or A (adjusted) data mode. R-mode profiles return None.

    Parameters
    ----------
    ds : xarray.Dataset
        Open Sprof NetCDF dataset.
    param : str
        Argo parameter name, e.g. "CHLA", "BBP700", "TEMP", "PSAL", "NITRATE".
    prof_idx : int
        Index along N_PROF.

    Returns
    -------
    (values, qc_flags, pressure) : tuple of np.ndarray, or (None, None, None)
    """
    if "STATION_PARAMETERS" not in ds.variables:
        return None, None, None

    def cs(x):
        return x.decode("utf-8", errors="ignore").strip() if isinstance(x, bytes) else str(x).strip()

    sp = [cs(p) for p in ds["STATION_PARAMETERS"].values[prof_idx]]
    if param not in sp:
        return None, None, None

    mode = cs(ds["PARAMETER_DATA_MODE"].values[prof_idx][sp.index(param)])
    if mode not in ("D", "A"):
        return None, None, None

    vk, qk = f"{param}_ADJUSTED", f"{param}_ADJUSTED_QC"
    pk = "PRES_ADJUSTED" if "PRES_ADJUSTED" in ds.variables else "PRES"
    if vk not in ds.variables or qk not in ds.variables:
        return None, None, None

    return ds[vk].values[prof_idx], ds[qk].values[prof_idx], ds[pk].values[prof_idx]


def get_monsoon_phase(month):
    """
    Classify a calendar month into a Northern Indian Ocean monsoon phase.

    Parameters
    ----------
    month : int
        Calendar month (1-12).

    Returns
    -------
    str
        "NE Monsoon" (Dec-Feb), "Spring Intermonsoon" (Mar-May),
        "SW Monsoon" (Jun-Sep), "Fall Intermonsoon" (Oct-Nov), or "Unknown".
    """
    if pd.isna(month):
        return "Unknown"
    if month in (12, 1, 2):   return "NE Monsoon"
    if month in (3, 4, 5):    return "Spring Intermonsoon"
    if month in (6, 7, 8, 9): return "SW Monsoon"
    if month in (10, 11):     return "Fall Intermonsoon"
    return "Unknown"


def round_half_up(value, decimals=1):
    """
    Round to a fixed number of decimal places using round-half-up convention.

    Python's built-in round() uses banker's rounding, which can silently
    exclude a ratio like 1.45 from a 1.5 threshold. This uses Decimal
    ROUND_HALF_UP so borderline values are classified as expected.

    Parameters
    ----------
    value : float
        Number to round. NaN/None is returned unchanged as np.nan.
    decimals : int, default 1
        Number of decimal places.

    Returns
    -------
    float
        Rounded value, or np.nan.
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    quant = Decimal("1").scaleb(-decimals)
    return float(Decimal(str(value)).quantize(quant, rounding=ROUND_HALF_UP))

In [ ]:
def calc_mld_teos10(pres, temp, psal, lon, lat):
    """
    Compute Mixed Layer Depth using a TEOS-10 potential-density (sigma0)
    threshold of MLD_DENSITY_THRESHOLD kg/m³ referenced to 10 dbar, with
    linear interpolation between the two bracketing samples.

    Returns the shallowest valid sample depth if the threshold is already
    exceeded there, and NaN when the threshold is never exceeded anywhere
    in the profile.

    Parameters
    ----------
    pres : np.ndarray
        Pressure (dbar).
    temp : np.ndarray
        In-situ temperature (°C), same length as pres.
    psal : np.ndarray
        Practical salinity interpolated onto the TEMP pressure grid.
    lon, lat : float
        Profile coordinates for gsw TEOS-10 conversions.

    Returns
    -------
    float
        MLD in dbar (~= m), or NaN (< 3 valid samples, shallowest sample
        deeper than 20 dbar, or density threshold never exceeded).
    """
    if len(pres) < 3:
        return np.nan

    SA    = gsw.SA_from_SP(psal, pres, lon, lat)
    CT    = gsw.CT_from_t(SA, temp, pres)
    sig0  = gsw.sigma0(SA, CT)

    ok = ~np.isnan(pres) & ~np.isnan(sig0)
    p, sig = pres[ok], sig0[ok]
    if len(p) < 3 or p[0] > 20:
        return np.nan

    sig_10 = np.interp(10.0, p, sig)
    diff   = np.abs(sig - sig_10)

    if not (diff > MLD_DENSITY_THRESHOLD).any():
        return np.nan

    idx = np.argmax(diff > MLD_DENSITY_THRESHOLD)
    if idx == 0:
        return float(p[0])

    p0, p1 = p[idx - 1], p[idx]
    d0, d1 = diff[idx - 1], diff[idx]
    if d1 == d0:
        return float(p1)
    return float(p0 + (MLD_DENSITY_THRESHOLD - d0) * (p1 - p0) / (d1 - d0))


def find_nitracline_depth(pres, nitrate):
    """
    Find nitracline depth: shallowest depth where nitrate first exceeds
    the surface-layer mean by NITRACLINE_DELTA µmol/m³ (Cornec et al. 2021).

    Surface reference layer is 0–NITRACLINE_SURFACE_DEPTH m.
    Depth is linearly interpolated between the bracketing samples.

    Parameters
    ----------
    pres, nitrate : np.ndarray
        Pressure (dbar) and nitrate concentration for this profile.

    Returns
    -------
    float
        Nitracline depth (dbar), or NaN if undetermined.
    """
    ok = ~np.isnan(pres) & ~np.isnan(nitrate)
    p, n = pres[ok], nitrate[ok]
    if len(p) < 2:
        return np.nan

    surf = p <= NITRACLINE_SURFACE_DEPTH
    if not surf.any():
        return np.nan

    threshold = np.mean(n[surf]) + NITRACLINE_DELTA
    cond = n >= threshold
    idx  = np.argmax(cond)

    if idx == 0:
        return float(p[0]) if cond[0] else np.nan

    p0, p1 = p[idx - 1], p[idx]
    n0, n1 = n[idx - 1], n[idx]
    if n1 == n0:
        return float(p1)
    return float(p0 + (threshold - n0) * (p1 - p0) / (n1 - n0))


def linear_interp_depth(pres, vals, threshold, trigger="above"):
    """
    Interpolate the depth where a parameter first crosses a threshold.

    Used for the 26 °C isotherm: trigger="below", threshold=26.0, which
    returns the depth where temperature first drops to or below 26 °C.

    Parameters
    ----------
    pres, vals : np.ndarray
        Pressure (dbar) and parameter values.
    threshold : float
        Crossing value.
    trigger : {"above", "below"}, default "above"
        "above": first depth where vals >= threshold.
        "below": first depth where vals <= threshold.

    Returns
    -------
    float
        Interpolated crossing depth (dbar), or NaN if not found.
    """
    ok = ~np.isnan(pres) & ~np.isnan(vals)
    p, v = pres[ok], vals[ok]
    if len(p) < 2:
        return np.nan

    cond = (v >= threshold) if trigger == "above" else (v <= threshold)
    idx  = np.argmax(cond)

    if idx == 0:
        return float(p[0]) if cond[0] else np.nan

    p0, p1 = p[idx - 1], p[idx]
    v0, v1 = v[idx - 1], v[idx]
    if v1 == v0:
        return float(p1)
    return float(p0 + (threshold - v0) * (p1 - p0) / (v1 - v0))

In [ ]:
def apply_smoothing(values, native_res, param_type):
    """
    Apply resolution-dependent rolling filter (Cornec et al. 2021).

    Smoothing is skipped when native_res is NaN or coarser than
    SMOOTHING_RES_THRESHOLD (3 m) to avoid over-smoothing coarse profiles.

    Parameters
    ----------
    values : np.ndarray
        Raw parameter values to smooth.
    native_res : float
        Mean vertical spacing (m) between samples in the top 250 m.
    param_type : {"CHLA", "BBP700"}
        CHLA: 5-point rolling median.
        BBP700: 5-point rolling median then 5-point rolling mean.

    Returns
    -------
    np.ndarray
        Smoothed values, or unchanged copy if native_res > threshold.
    """
    if np.isnan(native_res) or native_res > SMOOTHING_RES_THRESHOLD:
        return values.copy()

    s = pd.Series(values)
    if param_type == "CHLA":
        return s.rolling(5, center=True, min_periods=1).median().to_numpy()
    med = s.rolling(5, center=True, min_periods=1).median()
    return med.rolling(5, center=True, min_periods=1).mean().to_numpy()


def classify_profile_dcm(pres_chla, chla_raw, pres_bbp, bbp_raw):
    """
    Classify one BGC-Argo profile as NO, DAM, or DBM.

    Uses the revised 1.5x CHLA threshold with round-half-up rounding and a
    1.3x BBP700 threshold for the DAM/DBM split.

    Algorithm
    ---------
    1. Smooth CHLA if native vertical resolution ≤ 3 m (5-point rolling median).
    2. Find subsurface CHLA maximum on smoothed profile (search to 300 m depth);
       snap back to nearest raw sample for reported depth and value.
    3. chla_ratio = raw_CHLA_at_peak / median(raw_CHLA in 0–15 m surface layer).
       Round to 1 decimal (round-half-up). chla_ratio < 1.5 → "NO".
    4. For confirmed DCM: smooth BBP700, search a ±20 m window around DCM depth.
       bbp_ratio = raw_BBP700_at_window_peak / min(raw_BBP700 in 0–15 m).
       Round to 1 decimal (round-half-up). bbp_ratio > 1.3 → "DBM", else "DAM".

    Parameters
    ----------
    pres_chla, chla_raw : np.ndarray
        QC-filtered, depth-sorted pressure and CHLA.
    pres_bbp, bbp_raw : np.ndarray
        QC-filtered, depth-sorted pressure and BBP700.

    Returns
    -------
    dcm_type : str
        "NO", "DAM", or "DBM".
    z_dcm : float
        DCM depth (dbar), or NaN if NO.
    chla_dcm : float
        Raw CHLA at subsurface maximum, or NaN if NO.
    bbp_dcm : float
        Raw BBP700 at the subsurface window peak, or NaN if not evaluated.
    chla_ratio : float
        Rounded CHLA ratio used for classification, or NaN if uncomputable.
    bbp_ratio : float
        Rounded BBP700 ratio used for DAM/DBM split, or NaN if not evaluated.
    """
    nan6 = ("NO", np.nan, np.nan, np.nan, np.nan, np.nan)

    if len(pres_chla) < 5 or len(pres_bbp) < 2:
        return nan6

    top250_c = pres_chla <= TOP_250M_DEPTH
    if top250_c.sum() < 2:
        return nan6

    native_res_c = float(np.mean(np.diff(pres_chla[top250_c])))
    c_smooth     = apply_smoothing(chla_raw, native_res_c, "CHLA")

    search_c = pres_chla <= CHLA_SEARCH_MAX_DEPTH
    if not search_c.any():
        return nan6

    z_peak_smooth = pres_chla[search_c][np.argmax(c_smooth[search_c])]
    snap_c        = np.argmin(np.abs(pres_chla - z_peak_smooth))
    z_dcm         = pres_chla[snap_c]
    chla_dcm      = chla_raw[snap_c]

    surf_c = pres_chla <= DCM_SURFACE_DEPTH
    if not surf_c.any():
        return nan6

    surf_median = np.median(chla_raw[surf_c])
    if surf_median <= 0 or np.isnan(surf_median):
        return nan6

    chla_ratio = round_half_up(chla_dcm / surf_median, 1)
    if chla_ratio < DCM_CHLA_THRESHOLD:
        return "NO", np.nan, np.nan, np.nan, chla_ratio, np.nan

    top250_b  = pres_bbp <= TOP_250M_DEPTH
    nat_res_b = float(np.mean(np.diff(pres_bbp[top250_b]))) if top250_b.sum() > 1 else np.nan
    b_smooth  = apply_smoothing(bbp_raw, nat_res_b, "BBP700")

    window_b = (pres_bbp >= z_dcm - BBP_SEARCH_WINDOW) & (pres_bbp <= z_dcm + BBP_SEARCH_WINDOW)
    surf_b   = pres_bbp <= DCM_SURFACE_DEPTH

    if not window_b.any() or not surf_b.any():
        return "DAM", z_dcm, chla_dcm, np.nan, chla_ratio, np.nan

    z_bbp_smooth = pres_bbp[window_b][np.argmax(b_smooth[window_b])]
    snap_b       = np.argmin(np.abs(pres_bbp - z_bbp_smooth))
    bbp_dcm      = bbp_raw[snap_b]

    b_min_surf  = float(np.min(bbp_raw[surf_b]))
    if b_min_surf > 0:
        bbp_ratio_raw = bbp_dcm / b_min_surf
    else:
        bbp_ratio_raw = np.inf if bbp_dcm > 0 else 0.0

    bbp_ratio = round_half_up(bbp_ratio_raw, 1)
    dcm_type  = "DBM" if bbp_ratio > DBM_BBP_THRESHOLD else "DAM"
    return dcm_type, z_dcm, chla_dcm, bbp_dcm, chla_ratio, bbp_ratio

In [ ]:
def get_valid_qc_data(ds, param, prof_idx):
    """
    Return depth-sorted, QC-filtered (values, pressure) arrays for one parameter.

    Parameters
    ----------
    ds : xarray.Dataset
        Open Sprof NetCDF dataset.
    param : str
        Argo parameter name.
    prof_idx : int
        Profile index along N_PROF.

    Returns
    -------
    (pres, values) : tuple of np.ndarray
        Depth-sorted, QC-passed arrays. Both empty arrays if unavailable.
    """
    vals, qcs, p_raw = extract_var_with_fallback(ds, param, prof_idx)
    if vals is None:
        return np.array([]), np.array([])

    qcs_num = np.array([qc_to_int(q) for q in qcs])
    ok = ~np.isnan(p_raw) & ~np.isnan(vals) & (p_raw >= 0) & np.isin(qcs_num, list(QC_PASS_SET))
    p, v = p_raw[ok], vals[ok]
    idx  = np.argsort(p)
    return p[idx], v[idx]


def process_profile(ds, prof_idx):
    """
    Compute all variables for one BGC-Argo profile from an open Sprof dataset.

    Profiles without both CHLA and BBP700 in D/A mode are skipped entirely.
    Physical variables (MLD, nitracline, isotherm) are computed once regardless
    of the DCM classification result.

    Parameters
    ----------
    ds : xarray.Dataset
        Open Sprof NetCDF dataset.
    prof_idx : int
        Profile index along N_PROF.

    Returns
    -------
    dict or None
        Record with keys: WMO_ID, cycle, date, lat, lon, monsoon_phase,
        DCM_TYPE, DCM_Depth, Chla_DCM, bbp_DCM, chla_ratio, bbp_ratio,
        MLD, nitracline_depth, isotherm_26C_depth. None if CHLA or BBP700
        is unavailable in D/A mode.
    """
    p_c, c_raw = get_valid_qc_data(ds, "CHLA",    prof_idx)
    p_b, b_raw = get_valid_qc_data(ds, "BBP700",  prof_idx)
    if len(p_c) == 0 or len(p_b) == 0:
        return None

    p_t, t_raw = get_valid_qc_data(ds, "TEMP",    prof_idx)
    p_s, s_raw = get_valid_qc_data(ds, "PSAL",    prof_idx)
    p_n, n_raw = get_valid_qc_data(ds, "NITRATE", prof_idx)

    plat   = ds["PLATFORM_NUMBER"].values[prof_idx]
    wmo_id = plat.decode("utf-8").strip() if isinstance(plat, bytes) else str(plat).strip()

    cycle = np.nan
    if "CYCLE_NUMBER" in ds.variables:
        cv = ds["CYCLE_NUMBER"].values[prof_idx]
        cycle = int(cv) if not pd.isna(cv) else np.nan

    juld   = ds["JULD"].values[prof_idx]
    dt     = pd.to_datetime(juld) if not pd.isna(juld) else None
    date_s = dt.strftime("%Y-%m-%d %H:%M:%S") if dt else ""
    phase  = get_monsoon_phase(dt.month) if dt else "Unknown"

    lat = float(ds["LATITUDE"].values[prof_idx])
    lon = float(ds["LONGITUDE"].values[prof_idx])

    mld = np.nan
    if len(p_t) > 0 and len(p_s) > 0:
        s_interp = s_raw if np.array_equal(p_t, p_s) else np.interp(p_t, p_s, s_raw, left=np.nan, right=np.nan)
        mld = calc_mld_teos10(p_t, t_raw, s_interp, lon, lat)

    nitracline = find_nitracline_depth(p_n, n_raw)
    isotherm   = linear_interp_depth(p_t, t_raw, ISOTHERM_THRESHOLD, trigger="below")

    dcm_type, z_dcm, chla_dcm, bbp_dcm, chla_ratio, bbp_ratio = classify_profile_dcm(
        p_c, c_raw, p_b, b_raw
    )

    return {
        "WMO_ID": wmo_id, "cycle": cycle, "date": date_s,
        "lat": lat, "lon": lon, "monsoon_phase": phase,
        "DCM_TYPE": dcm_type, "DCM_Depth": z_dcm,
        "Chla_DCM": chla_dcm, "bbp_DCM": bbp_dcm,
        "chla_ratio": chla_ratio, "bbp_ratio": bbp_ratio,
        "MLD": mld, "nitracline_depth": nitracline,
        "isotherm_26C_depth": isotherm,
    }


def run_pipeline(float_dirs):
    """
    Walk all Sprof .nc files in float_dirs and process every profile.

    Parameters
    ----------
    float_dirs : list of str
        Directories to search recursively for *.nc Sprof files.

    Returns
    -------
    pandas.DataFrame
        One row per valid profile with all computed variables.
    """
    nc_files = []
    for d in float_dirs:
        nc_files.extend(glob.glob(os.path.join(d, "**/*.nc"), recursive=True))
    print(f"Found {len(nc_files)} NetCDF files. Processing...")

    records = []
    for path in nc_files:
        try:
            ds = xr.open_dataset(path)
            for i in range(ds.sizes.get("N_PROF", 0)):
                rec = process_profile(ds, i)
                if rec is not None:
                    records.append(rec)
            ds.close()
        except Exception as e:
            print(f"  Skipped {os.path.basename(path)}: {e}")

    df = pd.DataFrame(records)
    print(f"Done. {len(df)} valid profiles from {len(nc_files)} files.")
    return df

In [ ]:
def assign_basin(lat, lon):
    """
    Assign a sub-basin code from profile coordinates using strict bounding boxes.

    SCTR is checked first because its box is a geographic subset of SIO.

    Parameters
    ----------
    lat, lon : float
        Profile latitude and longitude.

    Returns
    -------
    str or None
        Sub-basin code: "SCTR", "AS", "BoB", "EIO", or "SIO".
        None if the profile falls outside all defined boxes.
    """
    for code, (lat_min, lat_max, lon_min, lon_max) in SUB_BASIN_BOXES.items():
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return code
    return None


def apply_geographic_filter(df):
    """
    Assign basin_box from coordinates and drop profiles outside defined boxes.

    Re-assigns basin purely from lat/lon, ignoring any folder-based label
    used during data download, to ensure geographic consistency.

    Parameters
    ----------
    df : pandas.DataFrame
        Pipeline output with lat and lon columns.

    Returns
    -------
    pandas.DataFrame
        Filtered copy with a new basin_box column; profiles outside all
        SUB_BASIN_BOXES are dropped.
    """
    df = df.copy()
    df["basin_box"] = df.apply(lambda r: assign_basin(r["lat"], r["lon"]), axis=1)
    n_out = df["basin_box"].isna().sum()
    df    = df[df["basin_box"].notna()].reset_index(drop=True)
    print(f"Basin assignment: dropped {n_out} profiles outside all boxes, kept {len(df)}.")
    print(df["basin_box"].value_counts().to_string())
    return df

In [ ]:
def standardize_dims(ds):
    """
    Rename common lat/lon/time dimension aliases to lat, lon, time.

    Parameters
    ----------
    ds : xarray.Dataset
        Satellite NetCDF dataset.

    Returns
    -------
    xarray.Dataset
        Dataset with standardized dimension names.
    """
    rmap = {}
    for c in ("latitude", "Latitude", "LAT"):
        if c in ds.dims or c in ds.coords: rmap[c] = "lat"
    for c in ("longitude", "Longitude", "LON"):
        if c in ds.dims or c in ds.coords: rmap[c] = "lon"
    for c in ("TIME", "Time"):
        if c in ds.dims or c in ds.coords: rmap[c] = "time"
    return ds.rename(rmap) if rmap else ds


def guess_var_name(ds, candidates):
    """
    Auto-detect the primary 3-D variable in a satellite NetCDF dataset.

    Parameters
    ----------
    ds : xarray.Dataset
        Opened and standardized satellite dataset.
    candidates : list of str
        Preferred variable names to check first (in order).

    Returns
    -------
    str
        Matched variable name.

    Raises
    ------
    ValueError
        If no 3-D variable is found among candidates or data_vars.
    """
    for c in candidates:
        if c in ds.data_vars:
            return c
    for v in ds.data_vars:
        if ds[v].ndim == 3:
            return v
    raise ValueError(f"No 3-D variable found. Available: {list(ds.data_vars)}")


def match_satellite(df, nc_path, candidates, col_name):
    """
    Add a satellite-matched column to df using nearest-neighbour selection
    in both time (calendar month) and space (lat/lon grid cell).

    Each profile is matched to the nearest available month and the nearest
    grid cell in the satellite NetCDF. Missing values become NaN.

    Parameters
    ----------
    df : pandas.DataFrame
        Profile table with date, lat, lon columns.
    nc_path : str
        Path to a monthly gridded satellite NetCDF file (time × lat × lon).
    candidates : list of str
        Variable names to try; first match wins.
    col_name : str
        Name of the new column to add.

    Returns
    -------
    pandas.DataFrame
        Copy of df with col_name appended.
    """
    ds      = xr.open_dataset(nc_path)
    ds      = standardize_dims(ds)
    varname = guess_var_name(ds, candidates)
    print(f"  {col_name}: variable '{varname}' from {os.path.basename(nc_path)}")

    df   = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    values = []
    for _, row in df.iterrows():
        try:
            val = ds[varname].sel(
                time=row["date"], lat=row["lat"], lon=row["lon"], method="nearest"
            ).values.item()
        except Exception:
            val = np.nan
        values.append(val)

    df[col_name] = values
    ds.close()

    n_miss = sum(np.isnan(v) if not isinstance(v, str) else False for v in values)
    print(f"  Matched {len(values) - n_miss}/{len(values)} profiles (missing: {n_miss}).")
    return df

In [ ]:
df = run_pipeline(FLOAT_DIRS)

df = apply_geographic_filter(df)

print("\nMatching satellite surface_chl...")
df = match_satellite(df, CHL_NC,
                     ["chl", "chlor_a", "CHL", "chlorophyll"], "surface_chl")

print("\nMatching satellite PAR...")
df = match_satellite(df, PAR_NC,
                     ["par", "PAR", "photosynthetically_available_radiation"], "PAR")

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved: {OUTPUT_CSV}  ({len(df)} profiles, {len(df.columns)} columns)")
print("\nDCM_TYPE counts:")
print(df["DCM_TYPE"].value_counts().to_string())
print("\nBasin counts:")
print(df["basin_box"].value_counts().to_string())